**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# RLS & Recursive Estimation

The rung between [APA](./Intro_AdFilt_APA.ipynb) and the [Kalman filter](./Intro_AdFilt_KF.ipynb): Recursive Least Squares solves the *entire* least-squares problem at every sample — exactly, recursively, without ever re-inverting a matrix — and turns out to be a Kalman filter wearing a different hat.

## 1. Pre-requisites

- [Adaptive Filtering: APA](./Intro_AdFilt_APA.ipynb) (the setup and its notation).
- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S2 (normal equations).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# same system-identification scenario as the APA workshop
M = 16
w_true = np.exp(-0.4*np.arange(M)) * np.cos(0.9*np.arange(M)); w_true /= np.linalg.norm(w_true)
N = 3000
from scipy import signal as sig
x = sig.lfilter([1.0], [1.0, -0.9], rng.standard_normal(N))     # correlated input (the hard case)
d = np.convolve(x, w_true)[:N] + 0.01*rng.standard_normal(N)

---
### 🕐 Session 1 of 2 — *Exponentially-Weighted Least Squares* (~35 min)
**Goal:** derive the RLS recursion from the matrix inversion lemma; implement it.
**Builds on:** [APA](./Intro_AdFilt_APA.ipynb). &nbsp; **Feeds into:** Session 2 (RLS ↔ Kalman).

---

## 2. Solving ALL of History, Every Sample

💡 **Intuition.** LMS/NLMS/APA use a *window* of data per update. RLS is greedier: at time $n$ it wants the exact minimizer of **all** past errors, forgetting old data exponentially: $J_n = \sum_{k\le n} \lambda^{n-k} e_k^2$ with forget factor $\lambda \lesssim 1$. Naively that's a matrix solve per sample. The rescue is the **matrix inversion lemma**: a rank-one update to $R$ produces a rank-one update to $R^{-1}$ — so the inverse is *carried along* and each step costs $O(M^2)$, not $O(M^3)$.

**The recursion.** Carry $P_n \approx R_n^{-1}$:
$$\mathbf{k}_n = \frac{P_{n-1}\mathbf{x}_n}{\lambda + \mathbf{x}_n^T P_{n-1}\mathbf{x}_n} \qquad e_n = d_n - \mathbf{w}_{n-1}^T \mathbf{x}_n$$
$$\mathbf{w}_n = \mathbf{w}_{n-1} + \mathbf{k}_n e_n \qquad P_n = \lambda^{-1}\big(P_{n-1} - \mathbf{k}_n \mathbf{x}_n^T P_{n-1}\big)$$

Same heartbeat as ever — *new = old + gain × error* — but the gain $\mathbf{k}_n$ now carries the full curvature of history, so convergence is nearly immune to input correlation (no more $\kappa$ penalty from [Optimization S2](../Intro_Math/Optimization/Optimization.ipynb)).

In [ ]:

# YOUR CODE HERE


**What just happened.** Same input, same target, same 16 taps — and RLS lands a final weight error of **0.00137** against NLMS's **0.00430**, roughly 3× closer to the truth. But the steady-state number undersells the result; the convergence curves are where the story is. RLS drops to its floor within a couple of hundred samples while NLMS is still descending thousands of samples later.

The reason is the input, and it was chosen deliberately. `x` is white noise pushed through a strongly resonant one-pole filter, so successive samples are heavily correlated and the autocorrelation matrix $R$ is badly conditioned. Gradient methods feel that directly: NLMS takes steps along the negative gradient, and in an ill-conditioned quadratic bowl the gradient points mostly *across* the narrow valley rather than along it, so progress is throttled by the condition number $\kappa$.

RLS never pays that toll, because $\mathbf{k}_n = P_{n-1}\mathbf{x}_n / (\lambda + \mathbf{x}_n^\top P_{n-1}\mathbf{x}_n)$ carries $P \approx R^{-1}$ — the curvature of the very bowl being descended. Multiplying by the inverse Hessian turns the elongated valley back into a circular one, which is Newton's method against gradient descent, exactly as in [Optimization S2](../Intro_Math/Optimization/Optimization.ipynb). "RLS converges in about $2M$ samples regardless of input colour" is the practical form of that statement, and $2M = 32$ here.

None of this is free, and the trade is worth naming now because Session 2's table formalises it: RLS costs $O(M^2)$ per sample against NLMS's $O(M)$, and it carries an $M \times M$ matrix that can drift out of symmetry over long runs. At $M = 16$ nobody cares. At $M = 1024$, in a real echo canceller, that is the difference between shipping and not.

---
### 🕐 Session 2 of 2 — *RLS ↔ Kalman* (~35 min)
**Goal:** see RLS as a Kalman filter for a static state; know the cost/robustness trade-table.
**Builds on:** Session 1; [Kalman](./Intro_AdFilt_KF.ipynb).

---

## 3. The Identification

💡 **Intuition.** Stare at the RLS recursion next to the [Kalman equations](./Intro_AdFilt_KF.ipynb): they are the *same algorithm*. Model the weights as a **static hidden state** ($F = I$, $Q = 0$) observed through $d_n = \mathbf{x}_n^T \mathbf{w} + v_n$ (so $H = \mathbf{x}_n^T$ changes every step): the Kalman gain becomes $\mathbf{k}_n$, the covariance $P$ is RLS's $P$, and $\lambda < 1$ plays the role of process noise — a confession that the 'static' weights actually drift. One framework, three names: RLS (filtering), recursive least squares (statistics), Kalman with random regressors (control).

In [ ]:
# Verify the identification numerically: Kalman-with-static-state ≡ RLS (λ=1)

# YOUR CODE HERE


**What just happened.** Two functions written from different starting points — one derived from the matrix inversion lemma, one from Bayesian state estimation — produce weight vectors agreeing to **3.2e-15**.

That number deserves to be read carefully, because it is a much stronger claim than "these methods perform similarly." It is machine epsilon accumulated over 3000 sequential updates, each involving a matrix–vector product and a division. Two algorithms that merely converged to the same optimum would agree to maybe 1e-6 and would disagree *along the way*; these agree at every step, to round-off. RLS with $\lambda = 1$ and a Kalman filter over a static state are not analogous, not asymptotically equivalent — they are the same recursion with different variable names.

Line them up and the dictionary is exact: the Kalman gain $P\mathbf{x}/(\mathbf{x}^\top P \mathbf{x} + R)$ is RLS's $\mathbf{k}_n$ with the measurement noise $R$ playing $\lambda$'s part; the covariance update is RLS's $P$ update; the innovation $d_n - \mathbf{w}^\top\mathbf{x}_n$ is the a-priori error. The modelling assumptions that produce this are $F = I$ and $Q = 0$ — the hidden state is the weight vector, and it is assumed never to move.

**What the identity does not say.** Kalman remains strictly the more general object, and the equivalence is confined to the static-state corner of it. Give the state a dynamics model $F$ and a process noise $Q$ and you can track weights that genuinely move, with an uncertainty that means something in probability rather than serving as bookkeeping. Seen from there, RLS's forgetting factor $\lambda$ is a one-scalar approximation to a full $Q$ — cheap, effective, and blunt, since it discounts every direction of the weight space at the same rate whether or not that matches how the system actually drifts.

The ladder from LMS to Kalman is therefore one algorithm at five levels of self-knowledge, and every rung shares the heartbeat *new = old + gain × error*. What changes is only how much the gain knows: nothing (LMS), the input power (NLMS), a small window (APA), all of history's curvature (RLS), or an explicit model of how the world moves (Kalman).

### The Family Portrait

| | LMS | NLMS | APA-$K$ | RLS | Kalman |
|---|---|---|---|---|---|
| Cost/sample | $O(M)$ | $O(M)$ | $O(K^2M)$ | $O(M^2)$ | $O(M^2)$+model |
| Colored-input speed | ✗ | ✗ | ○ | ✓ | ✓ |
| Tracks drifting systems | ✓ | ✓ | ✓ | via $\lambda$ | via $Q$ (principled) |
| Needs a state model | – | – | – | – | **yes** |
| Numerical fragility | robust | robust | mild | $P$ can lose symmetry* | same, use square-root forms |

\*Production RLS/Kalman uses QR/square-root updates — the [SOS lesson](../Intro_DSP/Filter_Design.ipynb) again: factor, don't invert.

## 4. Conclusion

RLS = exact least squares carried recursively via the matrix inversion lemma = Kalman filtering a static state. The whole adaptive-filtering ladder is one algorithm at increasing levels of self-knowledge.

---
## Where next

- [Beyond Kalman](./Beyond_Kalman.ipynb) — when the state *moves nonlinearly*.
- [Kernel Methods](../Intro_Mach_Learn/Kernel_Methods.ipynb) — KRLS: this recursion in a feature space.